# Graph-EFM Temporal Shift — Reduced WeatherBench 2 64×32

**Probabilistic calibration transfer under temporal distribution shift**

Clones code from GitHub. Stores data, checkpoints, and outputs on Google Drive.

**Budget:** ~50 Colab compute units (L4 GPU default, A100 optional)


In [ ]:
import os, sys, subprocess, shutil, time

# ============================================================
# CONFIGURABLE — EDIT THESE BEFORE RUNNING
# ============================================================
GITHUB_REPO = "https://github.com/YOUR_USERNAME/leap-graph-efm-shift.git"
NEURAL_LAM_BRANCH = "prob_model_global"
NEURAL_LAM_REPO = "https://github.com/mllam/neural-lam.git"

RUN_PROFILE = "l4_core"          # "l4_core" or "a100_optional"
DO_PREPROCESS = True
DO_TRAIN = True
DO_EVALUATE = True
DO_PLOT = True

MAX_COMPUTE_UNITS = 50
RESERVED_UNITS = 10

# ============================================================
# Mount Drive (for data / checkpoints / outputs)
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/leap_project"
PROJECT_DIR = "/content/leap_project"
REPO_DIR = os.path.join(PROJECT_DIR, "leap-graph-efm-shift")
NLAM_DIR = os.path.join(PROJECT_DIR, "neural-lam-prob-model")

os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(PROJECT_DIR, exist_ok=True)

# ============================================================
# Clone our experiment repo
# ============================================================
if not os.path.exists(REPO_DIR):
    print(f"Cloning experiment repo: {GITHUB_REPO}")
    !git clone {GITHUB_REPO} {REPO_DIR} 2>&1 | tail -3
else:
    print("Experiment repo already cloned. Pulling latest...")
    %cd {REPO_DIR}
    !git pull 2>&1 | tail -3
    %cd {PROJECT_DIR}

# ============================================================
# Clone neural-lam (prob_model_global branch)
# ============================================================
if not os.path.exists(NLAM_DIR):
    print(f"Cloning neural-lam ({NEURAL_LAM_BRANCH})...")
    !git clone --branch {NEURAL_LAM_BRANCH} {NEURAL_LAM_REPO} {NLAM_DIR} 2>&1 | tail -3
else:
    print("neural-lam already cloned.")

# ============================================================
# Apply our patches (modified neural-lam source files)
# ============================================================
patches_src = os.path.join(REPO_DIR, "neural-lam-patches")
if os.path.exists(patches_src):
    for root, dirs, files in os.walk(patches_src):
        rel = os.path.relpath(root, patches_src)
        for f in files:
            src = os.path.join(root, f)
            dst = os.path.join(NLAM_DIR, rel, f)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)
    print("Patches applied to neural-lam.")
else:
    print("WARNING: neural-lam-patches/ not found — modified files not applied!")

# ============================================================
# Ensure output directories exist, symlink data
# ============================================================
for d in ["configs", "scripts"]:
    # Already in our repo, no need to copy
    pass

os.makedirs(os.path.join(PROJECT_DIR, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(PROJECT_DIR, "results"), exist_ok=True)
os.makedirs(os.path.join(PROJECT_DIR, "figures"), exist_ok=True)
os.makedirs(os.path.join(PROJECT_DIR, "data"), exist_ok=True)

# Symlink data dir into neural-lam for its expected path structure
repo_data = os.path.join(NLAM_DIR, "data")
os.makedirs(repo_data, exist_ok=True)

# Also add neural-lam to Python path
sys.path.insert(0, NLAM_DIR)

print(f"\nSetup complete.")
print(f"  Experiment repo: {REPO_DIR}")
print(f"  neural-lam:      {NLAM_DIR}")
print(f"  Drive root:      {DRIVE_ROOT}")


In [ ]:
print("Installing dependencies...")

!pip install -q numpy scipy matplotlib xarray zarr wandb pyyaml tqdm
!pip install -q pytorch-lightning torch-geometric==2.3.1
!pip install -q shapely networkx Cartopy pyproj tueplots
!pip install -q git+https://github.com/deepmind/graphcast.git 2>&1 | tail -2
!pip install -q parse dataclass-wizard

# Verify imports
from neural_lam import constants
print(f"neural-lam OK (default GRID_SHAPE: {constants.GRID_SHAPE})")
print("Dependencies installed.")


In [ ]:
import yaml
from neural_lam import constants

CONFIG_PATH = os.path.join(REPO_DIR, "configs", "wb2_shift_64x32_graph_efm.yaml")
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# Load experiment constants (overrides paper defaults)
constants.load_experiment_config(CONFIG_PATH)

# Set up data paths — keep data on Drive for persistence
data_name = cfg["dataset"]["name"]
drive_data = os.path.join(DRIVE_ROOT, "data", data_name)
local_data = os.path.join(PROJECT_DIR, "data", data_name)
nl_data = os.path.join(NLAM_DIR, "data", data_name)
os.makedirs(local_data, exist_ok=True)
os.makedirs(nl_data, exist_ok=True)

# Symlink local data into neural-lam's expected path
if not os.path.exists(os.path.join(nl_data, "fields.zarr")):
    # Try Drive cache first
    if os.path.exists(os.path.join(drive_data, "fields.zarr")):
        print("Data found on Drive. Symlinking...")
        for item in os.listdir(drive_data):
            src = os.path.join(drive_data, item)
            dst = os.path.join(local_data, item)
            if not os.path.exists(dst):
                os.symlink(src, dst)
            nl_dst = os.path.join(nl_data, item)
            if not os.path.exists(nl_dst):
                os.symlink(src, nl_dst)

print(f"Grid: {constants.GRID_SHAPE}, State dim: {constants.GRID_STATE_DIM}")
print(f"Variables: {constants.PARAM_NAMES_SHORT}")
print(f"Splits: train {cfg['splits']['train']}, val {cfg['splits']['val']}")
print(f"          id {cfg['splits']['id']}, ood {cfg['splits']['ood']}")
print(f"Run profile: {RUN_PROFILE}")


## Step 4: Data Preparation

Downloads WeatherBench 2 data if not cached on Drive, then runs preprocessing:
forcing, grid static features, hierarchical mesh, parameter statistics.


In [ ]:
if not DO_PREPROCESS:
    print("DO_PREPROCESS=False — skipping")
else:
    fields_zarr = os.path.join(local_data, "fields.zarr")

    if not os.path.exists(fields_zarr):
        if os.path.exists(os.path.join(drive_data, "fields.zarr")):
            print("Data cached on Drive. Copying to local disk...")
            shutil.copytree(drive_data, local_data, dirs_exist_ok=True)
            # Re-symlink into neural-lam
            for item in os.listdir(local_data):
                src = os.path.join(local_data, item)
                dst = os.path.join(nl_data, item)
                if not os.path.exists(dst):
                    os.symlink(src, dst)
        else:
            print("Downloading WeatherBench 2 data (10–30 min)...")
            %cd {PROJECT_DIR}
            !python {REPO_DIR}/scripts/download_wb2_data.py \
                --output {fields_zarr} \
                --time_start {cfg['splits']['train'][0]} \
                --time_end {cfg['splits']['ood'][1]} \
                --method xarray
            # Symlink into neural-lam
            for item in os.listdir(local_data):
                src = os.path.join(local_data, item)
                dst = os.path.join(nl_data, item)
                if not os.path.exists(dst):
                    os.symlink(src, dst)
            # Cache to Drive
            drive_data_ds = os.path.join(DRIVE_ROOT, "data", data_name)
            os.makedirs(drive_data_ds, exist_ok=True)
            if not os.path.exists(os.path.join(drive_data_ds, "fields.zarr")):
                shutil.copytree(local_data, drive_data_ds, dirs_exist_ok=True)
                print(f"Data cached to Drive: {drive_data_ds}")
    else:
        print("Data already on local disk.")

    # Also symlink graphs dir
    graphs_src = os.path.join(NLAM_DIR, "graphs")
    os.makedirs(graphs_src, exist_ok=True)

    # Run preprocessing (forcing, grid features, mesh, weights)
    print("\nRunning preprocessing...")
    %cd {NLAM_DIR}
    !python {REPO_DIR}/scripts/prepare_wb2_subset.py \
        --config {CONFIG_PATH} \
        --steps forcing,grid_features,mesh,parameter_weights

    print("Preprocessing complete.")


## Step 5: CPU Smoke Check

Verifies Graph-EFM forward pass and ensemble sampling produce correct shapes.


In [ ]:
import torch
from neural_lam.models.graph_efm import GraphEFM
from types import SimpleNamespace

print("Building Graph-EFM with random weights (CPU)...")

mc, fc, gc, ev = cfg["model"], cfg["forecast"], cfg["graph"], cfg["evaluation"]
args = SimpleNamespace(
    hidden_dim=mc["hidden_dim"], latent_dim=mc["latent_dim"],
    hidden_layers=mc["hidden_layers"],
    processor_layers=mc["decoder_processor_layers"],
    encoder_processor_layers=mc["encoder_processor_layers"],
    prior_processor_layers=mc["prior_processor_layers"],
    kl_beta=mc["kl_beta"], crps_weight=mc["crps_weight"],
    loss=mc["loss"], sample_obs_noise=mc["sample_obs_noise"],
    output_std=mc["output_std"], prior_dist=mc["prior_dist"],
    learn_prior=mc["learn_prior"],
    graph=gc["name"], dataset=cfg["dataset"]["name"],
    lr=cfg["training"]["phase_a"]["lr"],
    step_length=cfg["sampling"]["step_length"],
    eval_leads=fc["eval_leads"], ensemble_size=ev["ensemble_size"],
    n_example_pred=0, batch_size=1,
)

model = GraphEFM(args)
model.eval()
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

B, M, T = 1, 4, fc["eval_leads"]
N = constants.GRID_SHAPE[0] * constants.GRID_SHAPE[1]
D, F = constants.GRID_STATE_DIM, constants.GRID_FORCING_DIM

with torch.no_grad():
    traj, _ = model.sample_trajectories(
        torch.randn(B, 2, N, D), torch.randn(B, T, N, F),
        torch.randn(B, T, N, D), M,
    )
assert traj.shape == (B, M, T, N, D), f"Shape mismatch: {traj.shape}"
print(f"Trajectory shape: {traj.shape}  PASSED")


## Step 6: GPU Setup & Memory Benchmark


In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type."

gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_mem / 1024**3
print(f"GPU: {gpu}  ({vram:.1f} GB)")

model = model.to("cuda")
B, T = cfg["training"]["phase_a"]["batch_size"], cfg["forecast"]["eval_leads"]
N = constants.GRID_SHAPE[0] * constants.GRID_SHAPE[1]
D, F = constants.GRID_STATE_DIM, constants.GRID_FORCING_DIM

init = torch.randn(B, 2, N, D, device="cuda")
targ = torch.randn(B, T, N, D, device="cuda")
forc = torch.randn(B, T, N, F, device="cuda")

torch.cuda.reset_peak_memory_stats()
with torch.no_grad():
    _ = model.sample_trajectories(init, forc, targ, 2)
peak = torch.cuda.max_memory_allocated() / 1024**2
est = peak * 4  # ens=2 -> ens=8
print(f"Peak memory (batch={B}, ens=2): {peak:.0f} MB")
print(f"Estimated  (batch={B}, ens=8): {est:.0f} MB")
print("OK" if est < vram * 1024 * 0.85 else "  WARNING: may OOM")


## Step 7: Training — Phase A (ar_steps=1, 100 epochs)

Checkpoint saved to Google Drive so it survives Colab disconnects.


In [ ]:
if not DO_TRAIN:
    print("DO_TRAIN=False — skipping")
else:
    ckpt = os.path.join(PROJECT_DIR, "checkpoints", "best_phase_a.ckpt")
    if os.path.exists(ckpt):
        print(f"Phase A checkpoint exists: {ckpt}")
    else:
        print("Training Phase A (ar_steps=1)...")
        %cd {PROJECT_DIR}
        !python {REPO_DIR}/scripts/train_shift_model.py \
            --config {CONFIG_PATH} \
            --phase a \
            --max_minutes 120
        if os.path.exists(ckpt):
            drive_ckpt = os.path.join(DRIVE_ROOT, "checkpoints")
            os.makedirs(drive_ckpt, exist_ok=True)
            shutil.copy2(ckpt, os.path.join(drive_ckpt, "best_phase_a.ckpt"))
            print("Saved to Drive.")


## Step 8: Training — Phase B (ar_steps=4, 50 epochs)

Resumes from Phase A best checkpoint.


In [ ]:
if DO_TRAIN:
    ckpt_a = os.path.join(PROJECT_DIR, "checkpoints", "best_phase_a.ckpt")
    ckpt_b = os.path.join(PROJECT_DIR, "checkpoints", "best_phase_b.ckpt")

    if not os.path.exists(ckpt_a):
        ckpt_a = os.path.join(DRIVE_ROOT, "checkpoints", "best_phase_a.ckpt")
        if os.path.exists(ckpt_a):
            shutil.copy2(ckpt_a, os.path.join(PROJECT_DIR, "checkpoints", "best_phase_a.ckpt"))

    if os.path.exists(ckpt_b):
        print(f"Phase B checkpoint exists: {ckpt_b}")
    elif not os.path.exists(ckpt_a):
        print("ERROR: Phase A checkpoint not found. Train Phase A first.")
    else:
        print("Training Phase B (ar_steps=4)...")
        %cd {PROJECT_DIR}
        !python {REPO_DIR}/scripts/train_shift_model.py \
            --config {CONFIG_PATH} \
            --phase b \
            --resume {ckpt_a} \
            --max_minutes 120
        if os.path.exists(ckpt_b):
            drive_ckpt = os.path.join(DRIVE_ROOT, "checkpoints")
            os.makedirs(drive_ckpt, exist_ok=True)
            shutil.copy2(ckpt_b, os.path.join(drive_ckpt, "best_phase_b.ckpt"))
            print("Saved to Drive.")


## Step 9: Evaluation

Runs ensemble forecasts on validation, ID (2011–2015), OOD (2016–2022).
Fits calibration, computes bootstrap CIs, t2m anomalies.


In [ ]:
if not DO_EVALUATE:
    print("DO_EVALUATE=False — skipping")
else:
    ckpt_b = os.path.join(PROJECT_DIR, "checkpoints", "best_phase_b.ckpt")
    if not os.path.exists(ckpt_b):
        ckpt_b = os.path.join(DRIVE_ROOT, "checkpoints", "best_phase_b.ckpt")
    if not os.path.exists(ckpt_b):
        raise RuntimeError(f"Checkpoint not found: {ckpt_b}")

    results_dir = os.path.join(PROJECT_DIR, "results")
    os.makedirs(results_dir, exist_ok=True)

    %cd {PROJECT_DIR}
    !python {REPO_DIR}/scripts/evaluate_shift.py \
        --config {CONFIG_PATH} \
        --checkpoint {ckpt_b} \
        --ensemble_size {cfg['evaluation']['ensemble_size']} \
        --output {results_dir} \
        --device cuda

    # Save to Drive
    drive_res = os.path.join(DRIVE_ROOT, "results")
    if os.path.exists(drive_res):
        shutil.rmtree(drive_res)
    shutil.copytree(results_dir, drive_res)

    # Quick summary
    import csv
    mp = os.path.join(results_dir, "metrics.csv")
    if os.path.exists(mp):
        with open(mp) as f:
            rows = list(csv.DictReader(f))
        for r in rows:
            if (r["split"] in ("id","ood") and r["calibration"]=="raw"
                    and r["variable"]=="t2m" and r["metric"]=="rmse"
                    and int(float(r["lead_hours"]))==24):
                print(f"  {r['split']} t2m RMSE@24h = {float(r['mean']):.3f}")
    print("Evaluation complete. Results on Drive.")


## Step 10: Figures

Generates 5 publication figures.


In [ ]:
if not DO_PLOT:
    print("DO_PLOT=False — skipping")
else:
    figs_dir = os.path.join(PROJECT_DIR, "figures")
    os.makedirs(figs_dir, exist_ok=True)

    %cd {PROJECT_DIR}
    !python {REPO_DIR}/scripts/plot_shift_results.py \
        --config {CONFIG_PATH} \
        --metrics {os.path.join(PROJECT_DIR, 'results')} \
        --output {figs_dir}

    from IPython.display import Image as IPImg, display
    for fn in sorted(os.listdir(figs_dir)):
        if fn.endswith(".pdf") or fn.endswith(".png"):
            print(f"\n--- {fn} ---")
            display(IPImg(filename=os.path.join(figs_dir, fn)))

    # Save to Drive
    drive_figs = os.path.join(DRIVE_ROOT, "figures")
    if os.path.exists(drive_figs):
        shutil.rmtree(drive_figs)
    shutil.copytree(figs_dir, drive_figs)
    print("Figures saved to Drive.")


##  Done!

Outputs on Google Drive (`MyDrive/leap_project/`):

| Path | Contents |
|------|----------|
| `data/` | Cached WeatherBench 2 zarr + preprocessing outputs |
| `checkpoints/best_phase_b.ckpt` | Trained Graph-EFM model |
| `results/metrics.csv` | Per-variable, per-lead, per-split metrics |
| `results/calibration_multipliers.json` | Fitted spread scaling factors |
| `results/bootstrap_ci.csv` | OOD−ID confidence intervals |
| `figures/01-05_*.pdf` | Five publication figures |

### Troubleshooting

- **Non-finite losses**: check data normalization, reduce lr
- **OOM**: reduce `batch_size` in config YAML (4 → 2 → 1)
- **Near-zero spread**: increase `kl_beta`, check prior network
- **Colab disconnect**: re-run; training resumes from Drive checkpoint
